# Exact Inference in Bayesian Networks

Now that we have a Bayesian network, what can we do with it? Well, since the Bayesian network is simply a representation of the full joint probability distribution we can do anything we would have done with the joint distribution. Any quantity that could be calculated from the joint distribution can be computed from the Bayesian network.

The straightforward way to do this would simply be to construct the full joint distribution directly, but that would be to miss the point. One of the reasons we use Bayesian networks is to reduce the number of parameters needed to represent the joint distribution and so to construct the full joint distribution from the network would be both wasteful and would also render the efficient and meaning representation provided by the network redundant.

Consider our example problem with the following DAG

 [DAG for calling the Fire Brigade](../images/fire_dag.png)

that represents the factorisation of the joint distribution as.

$P(C,R,D,A,S,F,T) = P(R)P(C)P(D)P(A\vert C)P(F\vert A)P(T\vert F,D) P(S\vert F,T)$

We have defined all of the probabilities in the terms in the factorisation and so we can compute the joint distribution. For convenience,

| $P(R)$ | $P(\lnot R)$ |
|:------:|:------------:|
| 0.3    | 0.7          |

</br></br>


| $P(C)$ | $P(\lnot C)$ |
|:------:|:------------:|
| 0.1    | 0.9          |

</br></br>



| $P(D)$ | $P(\lnot D)$ |
|:------:|:------------:|
| 0.5    | 0.5          |

</br></br>

|           | $P(A\vert C)$ | $P(\lnot A\vert C)$ |
|----------:|:-------------:|:-------------------:|
| $C$       | 0.01          |  0.99               |
| $\lnot C$ | 0.0           |  1.00               |

</br></br>


|           | $P(F\vert A)$ | $P(\lnot F\vert A)$ |
|----------:|:-------------:|:-------------------:|
| $A$       | 0.30          |  0.80               |
| $\lnot A$ | 0.001         |  0.990              |

</br></br>



|                  | $P(T\vert F,D)$ | $P(\lnot T\vert F,D)$ |
|-----------------:|:---------------:|:---------------------:|
| $D,F$            | 0.99            |  0.01                 |
| $D,\lnot F$      | 0.60            |  0.40                 |
| $\lnot D,F$      | 0.70            |  0.30                 |
| $\lnot D,\lnot F$| 0.01            |  0.99                |

</br></br>


|                  | $P(T\vert F,D)$ | $P(\lnot T\vert F,D)$ |
|-----------------:|:---------------:|:---------------------:|
| $T,F$            | 0.99            |  0.01                 |
| $T,\lnot F$      | 0.30            |  0.70                 |
| $\lnot D,F$      | 0.99            |  0.01                 |
| $\lnot D,\lnot F$| 0.001           |  0.999                |

**WARNING**: this is quite intricate and fiddly, but there are some well-define conventions we can use to make it simpler.

1. Use a sensible naming convention. We will use use:
- $P(A)\mapsto$`PA`
- $P(A\vert B)\mapsto$`PA_B`
- $P(A\vert B,C)\mapsto$`PA_BC`

2. Order the variables consistently. We will choose that
- Index zero corresponds to an outcome being False, and index one to True
- The indices are ordered in the same way as in the variable name, such that `PA_BC` has elements `PA_BC[ia][ib][ic]`

We then have

```PA = np.array([P(A=False),P(A=True)])```

```PA_B = np.array([[P(A=False|B=False),P(A=True|B=False)],[P(A=False|B=True),P(A=True|B=True)]])```

With these rules in mind we can write down our distributions

In [24]:
import numpy as np
PR = np.array([0.7,0.3])
PC = np.array([0.9,0.1])
PD = np.array([0.5,0.5])
PA_C = np.array([[1.0, 0.99],[0.0,0.01]])


At this point, take a step back and check that the way we have done this is correct with a manual calculation.

We now have anough information to calculate $P(A)$.

Since $P(A,C) = P(A\vert C)P(C)$ and $P(A) = \sum_C P(A,C)$ we have $P(A)=\sum_C P(A\vert C)P(C)$.

Let's check that this works with a manual calculation:

$P(\lnot A) = \sum_C P(\lnot A\vert C)P(C) = P(\lnot A\vert C)P(C) + P(\lnot A\vert \lnot C)P(\lnot C) = 0.99\times 0.1 + 1.0*0.9 = 0.999$

$P( A) = \sum_C P( A\vert C)P(C) = P( A\vert C)P(C) + P( A\vert \lnot C)P(\lnot C) = 0.01\times 0.1 + 0.0*0.9 = 0.001$

In [25]:
PA = (PA_C*PC).sum(axis=1)
print(PA)

[0.999 0.001]


In fact, there is an even easier way to do this: this is exactly equivalent to a matrix multiplication:

In [26]:
PA = PA_C@PC
print(PA)

[0.999 0.001]


So we have a nice neat notation here. Let's add a few more terms:

In [27]:
PF_A = np.array([[0.999, 0.7],[0.001,0.3]])
PT_FD = np.array([[[0.99,0.40],[0.30,0.01]],[[0.01,0.60],[0.70,0.99]]])


Pause again to check - we are now in a position to compute $P(T)$. Here we have to be careful with the order in which we multiple things. Let's compute one of these by hand just to make sure:

$P(T) = \sum_{CAF} P(T\vert F)P(F\vert A)P(A\vert C)P(C)$



In [28]:
# First compute P(F)
PF = PF_A@PA
print(PF)

# Then PT_F
PT_F = PT_FD@PD
print(PT_F)

# Finally PT
PT = PT_F @ PF
print(PT)

# Or all in one go:

PT = PT_FD @ PD @ PF_A @ PA
print(PT)

[0.998701 0.001299]
[[0.695 0.155]
 [0.305 0.845]]
[0.69429854 0.30570146]
[0.69429854 0.30570146]


Note that since we are working with matrices here, the order in which we compute things is crucial.

Is it right? The manual calculation is rather tedious so we just do a sanity check. This suggests that the cat is twice as likely to not be up the tree as it is to be up the tree. The chances of a house fire are very low, and so we should expect this to be dominated by presence/absence of the dog. We can use

$P(T) = \sum_D P(T\vert D)P(D) \approx \sum_D P(T\vert D,F=False)P(D)$

The distribution $P(T\vert D,F=False)$ is

|                  | $P(T\vert F=False,D)$ | $P(\lnot T\vert F=False,D)$ |
|-----------------:|:---------------:|:---------------------:|
| $D$      | 0.60            |  0.40                 |
| $\lnot D$| 0.01            |  0.99                 |


and so

$P(T) \approx P(T\vert\lnot D)P(\lnot D) + P(T\vert D)P(D) = 0.99\times 0.5 + 0.4\times 0.5) \approx 0.7$

and

$P(\lnot T) \approx P(T\vert\lnot D)P(\lnot D) + P(T\vert D)P(D) = 0.01\times 0.5 + 0.6\times 0.5) \approx 0.3$

Close enough to give us confidence that we have set this up right.

Finally, we can compute joint distributiuons from our conditional distributions. For example, $P(A,C)=P(A\vert C)P(C)$. How do we compute this? There is no summation required here so this is an element-wise multiplication:

In [29]:
print(PA_C)
print(PC)

PAC = PA_C*PC
print(PAC)

[[1.   0.99]
 [0.   0.01]]
[0.9 0.1]
[[0.9   0.099]
 [0.    0.001]]


This is normalised and so is a valid joint distribution. Is it correct?

$P(\lnot A,\lnot C) = P(\lnot A\vert \lnot C)P(\lnot C) = 1.0\times0.9 = 0.9$

$P(\lnot A, C) = P(\lnot A\vert C)P(C) = 0.99\times0.1 = 0.099$

$P(A,\lnot C) = P(A\vert \lnot C)P(\lnot C) = 0.0 \times0.099 = 0.0$

$P(A,C) = P(A\vert C)P(C) = 0.01\times0.1 = 0.001$